In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

path = "data/"
listings_clean = pd.read_csv(path + "listings_clean.csv")

# EDA

In [33]:
print(listings_clean[['price', 'accommodates', 'review_scores_rating']].describe())

print('\nroom_type counts:')
print(listings_clean['room_type'].value_counts())

print('\nproperty_type counts (top 10):')
print(listings_clean['property_type'].value_counts().head(10))

           price  accommodates  review_scores_rating
count 36613.0000    36613.0000            36613.0000
mean    197.6227        2.7695                3.2313
std     356.2521        1.9145                2.2296
min       8.0000        1.0000                0.0000
25%      85.0000        2.0000                0.0000
50%     150.0000        2.0000                4.6700
75%     203.0000        4.0000                4.9400
max   20000.0000       16.0000                5.0000

room_type counts:
room_type
Entire home/apt    19706
Private room       16237
Hotel room           491
Shared room          179
Name: count, dtype: int64

property_type counts (top 10):
property_type
Entire rental unit             15568
Private room in rental unit     9951
Private room in home            2532
Room in hotel                   1429
Entire home                     1304
Entire condo                    1107
Private room in townhouse       1087
Private room in condo            487
Entire loft               

In [34]:
# log transform price
listings_clean['price_log'] = np.log1p(listings_clean['price'])

# create has_reviews flag BEFORE imputing
listings_clean['has_reviews'] = (listings_clean['review_scores_rating'] > 0).astype(int)

# impute missing review scores with median from reviewed listings only
median_score = listings_clean[listings_clean['has_reviews'] == 1]['review_scores_rating'].median()
listings_clean.loc[listings_clean['has_reviews'] == 0, 'review_scores_rating'] = median_score

In [38]:
# null check
print('Null check:')
print(listings_clean[['price', 'price_log', 'review_scores_rating', 
                       'has_reviews', 'room_type', 
                       'neighbourhood_group_cleansed', 
                       'accommodates']].isnull().sum())
# price distribution
print(f'\nPrice log transform:')
print(listings_clean[['price', 'price_log']].describe())

# review score check
print(listings_clean['has_reviews'].value_counts())
print(listings_clean['review_scores_rating'].min())
print((listings_clean['review_scores_rating'] == 0).sum())

# review scores
print(f'\nMedian used for imputation: {median_score}')
print(f'Listings with reviews: {listings_clean["has_reviews"].sum()}')
print(f'Listings without reviews: {(listings_clean["has_reviews"] == 0).sum()}')
print(f'Imputed value check: {listings_clean[listings_clean["has_reviews"] == 0]["review_scores_rating"].unique()}')
print(f'\nFinal review score distribution:')
print(listings_clean["review_scores_rating"].describe())

Null check:
price                           0
price_log                       0
review_scores_rating            0
has_reviews                     0
room_type                       0
neighbourhood_group_cleansed    0
accommodates                    0
dtype: int64

Price log transform:
           price  price_log
count 36613.0000 36613.0000
mean    197.6227     4.9955
std     356.2521     0.6832
min       8.0000     2.1972
25%      85.0000     4.4543
50%     150.0000     5.0173
75%     203.0000     5.3181
max   20000.0000     9.9035
has_reviews
1    25018
0    11595
Name: count, dtype: int64
1.0
0

Median used for imputation: 4.86
Listings with reviews: 25018
Listings without reviews: 11595
Imputed value check: [4.86]

Final review score distribution:
count   36613.0000
mean        4.7704
std         0.3682
min         1.0000
25%         4.7500
50%         4.8600
75%         4.9400
max         5.0000
Name: review_scores_rating, dtype: float64
